# 🇮🇩 Indonesian Sentiment & Emotion Analysis
## Text Mining J1 | Informatika Unud

**Dataset:** PRDECT-ID - Product Reviews Dataset for Emotions Classification Tasks (Indonesian)  
**Source:** [Mendeley Data - DOI: 10.17632/574v66hf2v.1](https://data.mendeley.com/datasets/574v66hf2v/1)  
**License:** CC BY 4.0  
**Domain:** Tokopedia product reviews (29 categories, Bahasa Indonesia)  
**Labels:** Sentiment (Positive/Negative) + Emotion (Love, Happiness, Anger, Fear, Sadness)

---

### 🎯 Tujuan Portfolio
Notebook ini mendemonstrasikan kemampuan lengkap Text Mining pipeline untuk klasifikasi sentimen & emosi teks Bahasa Indonesia:

1. **Data Acquisition** - load dataset publik dari Mendeley
2. **Preprocessing** - NLP pipeline Bahasa Indonesia (case folding, tokenisasi, stopword, stemming)
3. **EDA** - eksplorasi distribusi label, panjang teks, word frequency
4. **Feature Engineering** - TF-IDF dan IndoBERT embedding
5. **Modeling** - Naive Bayes, SVM, dan fine-tuned IndoBERT
6. **Evaluation** - Accuracy, Precision, Recall, F1, Confusion Matrix
7. **Error Analysis** - deep dive ke prediksi salah

---
**Relevansi ke Skripsi:** Pipeline ini identik dengan yang dibutuhkan untuk *Sentiment Analysis Berita Finansial BEI* - bedanya hanya domain teks dan sumber data.

## 📦 0. Setup & Install Dependencies

In [1]:
# Install all required packages
%pip install -q transformers datasets PySastrawi gdown nltk scikit-learn \
               matplotlib seaborn wordcloud plotly pandas numpy tqdm

import warnings
warnings.filterwarnings('ignore')
print('✅ All packages installed!')

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed!



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import os
from collections import Counter
from tqdm.notebook import tqdm

# NLP
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# Transformer
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Style
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})
sns.set_theme(style='whitegrid', palette='muted')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('✅ Imports done!')
print(f'🖥️  GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   → {torch.cuda.get_device_name(0)}')

ModuleNotFoundError: No module named 'torch'

## 📥 1. Data Acquisition - PRDECT-ID (Mendeley Data)

In [ ]:
# ─────────────────────────────────────────────────────────────
# OPTION A: Download langsung dari Mendeley Data API
# Dataset DOI: 10.17632/574v66hf2v.1
# ─────────────────────────────────────────────────────────────

import urllib.request
import zipfile

MENDELEY_URL = 'https://data.mendeley.com/public-files/datasets/574v66hf2v/files/7e4b8c4c-a5dc-40d6-b5b4-e0a3ba84c555/file_downloaded'
ZIP_FILE = 'prdect_id.zip'

try:
    print('📡 Downloading PRDECT-ID from Mendeley Data...')
    urllib.request.urlretrieve(MENDELEY_URL, ZIP_FILE)
    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        z.extractall('prdect_id/')
    print('✅ Downloaded & extracted!')
except Exception as e:
    print(f'⚠️  Direct download failed: {e}')
    print('   → Trying fallback (gdown)...')
    # Fallback: gdown dari Google Drive mirror
    print('   → Gunakan OPTION B di bawah jika download gagal.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Load Dataset
# ─────────────────────────────────────────────────────────────

# Cari file CSV yang sudah didownload
import glob
csv_files = glob.glob('prdect_id/**/*.csv', recursive=True)
print('CSV files found:', csv_files)

# Load (sesuaikan path jika berbeda)
if csv_files:
    df = pd.read_csv(csv_files[0])
else:
    # Demo mode - simulasi struktur dataset jika file belum ada
    print('⚠️  File belum didownload. Running in DEMO mode dengan synthetic data.')
    DEMO_DATA = {
        'Review': [
            'Produknya bagus banget, pengiriman cepat, puas sekali!',
            'Barang tidak sesuai gambar, kecewa banget sama penjual ini',
            'Oke lah, biasa aja sih, tidak ada yang istimewa',
            'Suka banget! Kualitasnya melebihi ekspektasi saya',
            'Paket rusak waktu sampai, minta refund gak direspon',
            'Lumayan, harga sesuai kualitas, recommended deh',
            'Bahan bagus, jahitan rapi, nyaman dipakai sehari-hari',
            'Sangat mengecewakan, tidak sesuai deskripsi sama sekali',
            'Seller ramah dan fast response, produk original',
            'Takut awalnya tapi ternyata oke, bisa dipercaya',
        ] * 100,
        'Sentiment': (['Positive'] * 7 + ['Negative'] * 3) * 100,
        'Emotion': (['Love', 'Happiness', 'Happiness', 'Love', 'Anger', 'Happiness', 'Love', 'Anger', 'Happiness', 'Fear']) * 100,
        'Category': (['Fashion', 'Electronics', 'Food', 'Home', 'Electronics', 'Fashion', 'Food', 'Electronics', 'Home', 'Fashion']) * 100,
        'Customer Rating': np.random.choice([1,2,3,4,5], 1000),
    }
    df = pd.DataFrame(DEMO_DATA)

print(f'\n📊 Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Basic info
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

## 🔍 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── Identifikasi kolom teks dan label ──
# Sesuaikan nama kolom dengan struktur aktual PRDECT-ID
TEXT_COL   = 'Review'      # kolom teks review
SENT_COL   = 'Sentiment'   # Positive / Negative
EMOT_COL   = 'Emotion'     # Love / Happiness / Anger / Fear / Sadness

# Pastikan kolom ada
for col in [TEXT_COL, SENT_COL, EMOT_COL]:
    if col not in df.columns:
        print(f'⚠️  Kolom "{col}" tidak ditemukan. Sesuaikan nama kolom di atas.')
        print(f'   Kolom yang tersedia: {list(df.columns)}')

# Drop NaN di kolom utama
df = df.dropna(subset=[TEXT_COL, SENT_COL, EMOT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)

print(f'Jumlah data setelah cleaning: {len(df):,}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('PRDECT-ID - Label Distribution', fontsize=14, fontweight='bold', y=1.02)

# Sentiment distribution
sent_counts = df[SENT_COL].value_counts()
colors_sent = ['#2ecc71', '#e74c3c']
axes[0].pie(sent_counts.values, labels=sent_counts.index,
            autopct='%1.1f%%', colors=colors_sent, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Sentiment Distribution', fontweight='bold')

# Emotion distribution
emot_counts = df[EMOT_COL].value_counts()
colors_emot = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#2ecc71']
bars = axes[1].barh(emot_counts.index, emot_counts.values,
                     color=colors_emot[:len(emot_counts)], edgecolor='white')
axes[1].set_title('Emotion Distribution', fontweight='bold')
axes[1].set_xlabel('Count')
for bar, val in zip(bars, emot_counts.values):
    axes[1].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)

# Text length distribution
df['text_len'] = df[TEXT_COL].apply(lambda x: len(x.split()))
axes[2].hist(df['text_len'], bins=40, color='#3498db', edgecolor='white', alpha=0.8)
axes[2].axvline(df['text_len'].median(), color='red', linestyle='--', label=f'Median: {df["text_len"].median():.0f}')
axes[2].set_title('Review Length (words)', fontweight='bold')
axes[2].set_xlabel('Word Count')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'\n📊 Stats teks:\n{df["text_len"].describe().round(2)}')

In [ ]:
# Word Cloud per Sentiment
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Word Cloud - Positive vs Negative Reviews', fontsize=13, fontweight='bold')

for ax, label, cmap in zip(axes, ['Positive', 'Negative'], ['Greens', 'Reds']):
    subset = df[df[SENT_COL] == label][TEXT_COL]
    text = ' '.join(subset.values)
    wc = WordCloud(
        width=700, height=350,
        background_color='white',
        colormap=cmap,
        max_words=80,
        collocations=False
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{label} Reviews', fontweight='bold', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig('wordcloud.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Top 20 kata per sentimen (before preprocessing)
from sklearn.feature_extraction.text import CountVectorizer

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 20 Kata Paling Sering - Raw Text', fontsize=13, fontweight='bold')

for ax, label, color in zip(axes, ['Positive', 'Negative'], ['#2ecc71', '#e74c3c']):
    subset = df[df[SENT_COL] == label][TEXT_COL]
    vec = CountVectorizer(max_features=20, min_df=2)
    vec.fit_transform(subset)
    words = vec.get_feature_names_out()
    counts = vec.transform(subset).toarray().sum(axis=0)
    sorted_idx = np.argsort(counts)[::-1]
    ax.barh([words[i] for i in sorted_idx], [counts[i] for i in sorted_idx],
            color=color, alpha=0.85)
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('top_words.png', bbox_inches='tight', dpi=150)
plt.show()

## 🧹 3. Text Preprocessing - Indonesian NLP Pipeline

In [ ]:
# ── Build Indonesian NLP Pipeline ──

# Inisialisasi Sastrawi (Indonesian stemmer & stopword)
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopword_list = set(stopword_factory.get_stop_words())

# Tambah custom stopwords domain review produk
custom_stopwords = {
    'yg', 'dgn', 'utk', 'krn', 'kl', 'kalo', 'tp', 'tapi',
    'udah', 'udh', 'sdh', 'sudah', 'sudahh', 'bgt', 'banget',
    'nih', 'sih', 'deh', 'dong', 'lah', 'aja', 'aj', 'jg',
    'juga', 'sm', 'sama', 'ke', 'di', 'nya', 'nya', 'ini', 'itu',
    'ga', 'gak', 'tidak', 'tdk', 'nggak', 'enggak', 'blm', 'belum',
}
stopword_list = stopword_list.union(custom_stopwords)

# Slang dictionary (sebagian, bisa diperluas)
SLANG_DICT = {
    'gw': 'saya', 'gue': 'saya', 'lo': 'kamu', 'lu': 'kamu',
    'beli': 'beli', 'pake': 'pakai', 'makan': 'makan',
    'bgt': 'sangat', 'banget': 'sangat', 'bngt': 'sangat',
    'ok': 'baik', 'oke': 'baik', 'okelah': 'baik',
    'mantap': 'bagus', 'mantul': 'bagus', 'kece': 'bagus',
    'jelek': 'buruk', 'jelekk': 'buruk', 'jlek': 'buruk',
    'dapet': 'dapat', 'dpt': 'dapat',
    'tmn': 'teman', 'temen': 'teman',
    'dikirim': 'kirim', 'ngirim': 'kirim',
    'kpd': 'kepada', 'dgn': 'dengan', 'utk': 'untuk',
    'pdhl': 'padahal', 'bkn': 'bukan', 'sbg': 'sebagai',
}

def normalize_slang(text: str) -> str:
    words = text.split()
    return ' '.join([SLANG_DICT.get(w, w) for w in words])

def preprocess_indonesian(text: str, do_stem: bool = True) -> str:
    """Full Indonesian NLP preprocessing pipeline."""
    # 1. Lowercase
    text = text.lower()
    # 2. Hapus URL, mention, hashtag
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    # 3. Hapus angka dan karakter khusus
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 4. Normalisasi spasi
    text = re.sub(r'\s+', ' ', text).strip()
    # 5. Normalisasi slang
    text = normalize_slang(text)
    # 6. Tokenisasi
    tokens = word_tokenize(text)
    # 7. Hapus stopword
    tokens = [t for t in tokens if t not in stopword_list and len(t) > 2]
    # 8. Stemming (Sastrawi)
    if do_stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

# Test
sample = 'Produknya bagus banget!! pengiriman cepet, puas bgt sama seller yg ini 👍'
print(f'Original : {sample}')
print(f'Processed: {preprocess_indonesian(sample)}')

In [ ]:
# Apply preprocessing ke seluruh dataset
print('⏳ Preprocessing... (mungkin butuh 1-2 menit)')
tqdm.pandas()

df['text_clean'] = df[TEXT_COL].progress_apply(lambda x: preprocess_indonesian(x, do_stem=True))
df['text_nostem'] = df[TEXT_COL].progress_apply(lambda x: preprocess_indonesian(x, do_stem=False))

# Remove empty results
df = df[df['text_clean'].str.strip() != ''].copy()

print(f'\n✅ Preprocessing selesai!')
print(f'Total data: {len(df):,}')
print('\nContoh perbandingan:')
df[[TEXT_COL, 'text_clean']].head(3)

In [ ]:
# Visualisasi Before vs After Preprocessing
df['len_before'] = df[TEXT_COL].apply(lambda x: len(x.split()))
df['len_after']  = df['text_clean'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Perbandingan Panjang Teks: Before vs After Preprocessing', fontweight='bold')

axes[0].hist(df['len_before'], bins=40, alpha=0.7, label='Before', color='#3498db', edgecolor='white')
axes[0].hist(df['len_after'],  bins=40, alpha=0.7, label='After',  color='#e74c3c', edgecolor='white')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()
axes[0].set_title('Distribusi Panjang Teks')

reduction = ((df['len_before'] - df['len_after']) / df['len_before'] * 100)
axes[1].hist(reduction.clip(0, 100), bins=30, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[1].axvline(reduction.mean(), color='red', linestyle='--', label=f'Mean: {reduction.mean():.1f}%')
axes[1].set_xlabel('Reduksi Token (%)')
axes[1].set_ylabel('Frekuensi')
axes[1].set_title('Reduksi Token Setelah Preprocessing')
axes[1].legend()

plt.tight_layout()
plt.savefig('preprocessing_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'\n📉 Rata-rata reduksi token: {reduction.mean():.1f}%')

## ⚙️ 4. Feature Engineering

In [ ]:
# ── Encode Labels ──
le_sent = LabelEncoder()
le_emot = LabelEncoder()

df['label_sent'] = le_sent.fit_transform(df[SENT_COL])
df['label_emot'] = le_emot.fit_transform(df[EMOT_COL])

print('Sentiment classes:', le_sent.classes_)
print('Emotion  classes:', le_emot.classes_)

# ── Train/Val/Test Split (70/15/15) ──
X = df['text_clean'].values
y_sent = df['label_sent'].values
y_emot = df['label_emot'].values

X_train, X_temp, y_sent_train, y_sent_temp, y_emot_train, y_emot_temp = train_test_split(
    X, y_sent, y_emot, test_size=0.3, random_state=SEED, stratify=y_sent
)
X_val, X_test, y_sent_val, y_sent_test, y_emot_val, y_emot_test = train_test_split(
    X_temp, y_sent_temp, y_emot_temp, test_size=0.5, random_state=SEED, stratify=y_sent_temp
)

print(f'\n📂 Data splits:')
print(f'  Train : {len(X_train):,} samples')
print(f'  Val   : {len(X_val):,} samples')
print(f'  Test  : {len(X_test):,} samples')

In [ ]:
# ── TF-IDF Feature Extraction ──

# Unigram
tfidf_uni = TfidfVectorizer(
    ngram_range=(1, 1),
    max_features=15000,
    min_df=2,
    sublinear_tf=True
)

# Bigram
tfidf_bi = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000,
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf_uni = tfidf_uni.fit_transform(X_train)
X_val_tfidf_uni   = tfidf_uni.transform(X_val)
X_test_tfidf_uni  = tfidf_uni.transform(X_test)

X_train_tfidf_bi = tfidf_bi.fit_transform(X_train)
X_val_tfidf_bi   = tfidf_bi.transform(X_val)
X_test_tfidf_bi  = tfidf_bi.transform(X_test)

print(f'TF-IDF Unigram shape: {X_train_tfidf_uni.shape}')
print(f'TF-IDF Bigram  shape: {X_train_tfidf_bi.shape}')

# Visualisasi top TF-IDF terms
feature_names = np.array(tfidf_uni.get_feature_names_out())
mean_tfidf = X_train_tfidf_uni.mean(axis=0).A1
top_idx = mean_tfidf.argsort()[::-1][:20]

plt.figure(figsize=(10, 5))
plt.barh(feature_names[top_idx][::-1], mean_tfidf[top_idx][::-1], color='#3498db', alpha=0.85)
plt.title('Top 20 TF-IDF Features (Training Set)', fontweight='bold')
plt.xlabel('Mean TF-IDF Score')
plt.tight_layout()
plt.savefig('tfidf_features.png', bbox_inches='tight', dpi=150)
plt.show()

## 🤖 5. Modeling - Baseline ML Models (Sentiment)

In [ ]:
# ── Definisi model ──
models = {
    'Naive Bayes (Unigram)':     (MultinomialNB(alpha=0.5),     X_train_tfidf_uni, X_test_tfidf_uni),
    'Naive Bayes (Bigram)':      (MultinomialNB(alpha=0.5),     X_train_tfidf_bi,  X_test_tfidf_bi),
    'Linear SVM (Unigram)':      (LinearSVC(C=1.0, max_iter=2000), X_train_tfidf_uni, X_test_tfidf_uni),
    'Linear SVM (Bigram)':       (LinearSVC(C=1.0, max_iter=2000), X_train_tfidf_bi,  X_test_tfidf_bi),
    'Logistic Regression':       (LogisticRegression(C=1.0, max_iter=1000, random_state=SEED), X_train_tfidf_bi, X_test_tfidf_bi),
}

results = {}

for name, (model, X_tr, X_te) in models.items():
    model.fit(X_tr, y_sent_train)
    preds = model.predict(X_te)
    acc  = accuracy_score(y_sent_test, preds)
    f1   = f1_score(y_sent_test, preds, average='weighted')
    prec = precision_score(y_sent_test, preds, average='weighted', zero_division=0)
    rec  = recall_score(y_sent_test, preds, average='weighted', zero_division=0)
    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 (weighted)': f1, 'model': model, 'preds': preds}
    print(f'{name:<35} | Acc: {acc:.4f} | F1: {f1:.4f}')

print('\n✅ Baseline models selesai!')

In [ ]:
# ── Comparison Chart ──
results_df = pd.DataFrame({k: {m: v for m, v in v.items() if m not in ['model', 'preds']}
                           for k, v in results.items()}).T

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(results_df))
width = 0.2
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 (weighted)']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df.index, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0.5, 1.0)
ax.set_title('Model Comparison - Sentiment Classification', fontweight='bold', fontsize=12)
ax.set_ylabel('Score')
ax.legend(loc='lower right')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n📊 Results Summary:')
print(results_df[metrics].applymap(lambda x: f'{x:.4f}'))

In [ ]:
# ── Confusion Matrix - Best Model ──
best_model_name = results_df['F1 (weighted)'].idxmax()
best_preds = results[best_model_name]['preds']

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_sent_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_sent.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix - {best_model_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'\n🏆 Best Model: {best_model_name}')
print(f'   F1 Score  : {results_df.loc[best_model_name, "F1 (weighted)"]:.4f}')
print(f'   Accuracy  : {results_df.loc[best_model_name, "Accuracy"]:.4f}')
print('\n📋 Classification Report:')
print(classification_report(y_sent_test, best_preds, target_names=le_sent.classes_))

## 🤗 6. IndoBERT Fine-tuning (Advanced)

In [ ]:
# ── IndoBERT Setup ──
# IndoBERT: model BERT yang di-pretrain pada corpus Bahasa Indonesia
# Paper: https://arxiv.org/abs/2009.05387

MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LEN = 128
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test tokenizer
sample_tokens = tokenizer('produk bagus pengiriman cepat', return_tensors='pt')
print(f'Sample tokenization: {sample_tokens["input_ids"].shape}')
print('Tokens:', tokenizer.convert_ids_to_tokens(sample_tokens['input_ids'][0]))

In [ ]:
# ── Prepare HuggingFace Dataset ──
# Gunakan text_nostem untuk IndoBERT (biar BERT handle sendiri)

df_train = df.iloc[:len(X_train)].copy()
df_val   = df.iloc[len(X_train):len(X_train)+len(X_val)].copy()
df_test_ = df.iloc[len(X_train)+len(X_val):].copy()

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = Dataset.from_dict({'text': X_train.tolist(), 'labels': y_sent_train.tolist()})
val_ds   = Dataset.from_dict({'text': X_val.tolist(),   'labels': y_sent_val.tolist()})
test_ds  = Dataset.from_dict({'text': X_test.tolist(),  'labels': y_sent_test.tolist()})

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds   = val_ds.map(tokenize_function, batched=True)
test_ds  = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(['text'])
val_ds   = val_ds.remove_columns(['text'])
test_ds  = test_ds.remove_columns(['text'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')

print(f'Dataset ready: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

# Load IndoBERT for classification
NUM_LABELS = len(le_sent.classes_)
print(f'Loading IndoBERT for {NUM_LABELS}-class classification...')

indobert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True
)

# Training arguments
training_args = TrainingArguments(
    output_dir='./indobert_output',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=LEARNING_RATE,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # Mixed precision jika GPU tersedia
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=indobert_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('✅ Trainer siap!')

In [ ]:
# ── FINE-TUNING (jalankan jika punya GPU/TPU) ──
# ⚠️  Butuh GPU untuk runtime yang wajar (15-30 menit di T4)
# Di CPU bisa sangat lambat

if torch.cuda.is_available():
    print('🚀 GPU detected! Starting fine-tuning...')
    trainer.train()

    # Evaluasi pada test set
    print('\n📊 Evaluating on test set...')
    test_results = trainer.evaluate(test_ds)
    print(f'IndoBERT Test Accuracy: {test_results["eval_accuracy"]:.4f}')
    print(f'IndoBERT Test F1:       {test_results["eval_f1"]:.4f}')

    # Save model
    trainer.save_model('./indobert_sentiment_final')
    tokenizer.save_pretrained('./indobert_sentiment_final')
    print('\n💾 Model saved to ./indobert_sentiment_final')
else:
    print('⚠️  GPU tidak tersedia. Aktifkan Runtime GPU di Colab:')
    print('   Runtime → Change runtime type → T4 GPU')
    print('\n   IndoBERT fine-tuning di-skip. Menggunakan baseline ML result.')

## 🔬 7. Error Analysis

In [ ]:
# Analisis kesalahan prediksi dari model terbaik (baseline)
best_model_obj = results[best_model_name]['model']

# Ambil data test asli
test_texts_raw = df[TEXT_COL].values[-len(X_test):]
test_texts_clean = X_test
true_labels = le_sent.inverse_transform(y_sent_test)
pred_labels = le_sent.inverse_transform(best_preds)

error_df = pd.DataFrame({
    'original_text': test_texts_raw,
    'clean_text':    test_texts_clean,
    'true_label':    true_labels,
    'pred_label':    pred_labels,
    'correct':       true_labels == pred_labels
})

errors = error_df[~error_df['correct']].copy()
print(f'Total misclassified: {len(errors)} / {len(error_df)} ({len(errors)/len(error_df)*100:.1f}%)')
print(f'\nError breakdown:')
print(errors.groupby(['true_label', 'pred_label']).size().reset_index(name='count'))

print('\n🔍 Sample kesalahan prediksi:')
errors[['original_text', 'true_label', 'pred_label']].sample(min(5, len(errors)), random_state=SEED)

In [ ]:
# ── Analisis kata yang menyebabkan misclassifikasi ──

# False Positive: diprediksi Positive, padahal Negative
fp_texts = errors[errors['pred_label'] == 'Positive']['clean_text'].values
# False Negative: diprediksi Negative, padahal Positive  
fn_texts = errors[errors['pred_label'] == 'Negative']['clean_text'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
fig.suptitle('Word Cloud - Error Analysis', fontweight='bold', fontsize=12)

for ax, texts, title, cmap in [
    (axes[0], fp_texts, 'False Positive\n(Negatif → Diprediksi Positif)', 'YlOrRd'),
    (axes[1], fn_texts, 'False Negative\n(Positif → Diprediksi Negatif)', 'Blues'),
]:
    if len(texts) > 0:
        wc = WordCloud(width=600, height=300, background_color='white',
                       colormap=cmap, max_words=50, collocations=False)
        wc.generate(' '.join(texts))
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(title, fontweight='bold', fontsize=10)
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, 'No samples', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title, fontweight='bold', fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig('error_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 📈 8. Emotion Classification (Multi-class)

In [ ]:
# Klasifikasi 5 emosi: Love, Happiness, Anger, Fear, Sadness
emotion_models = {
    'Linear SVM':          LinearSVC(C=1.0, max_iter=3000),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, multi_class='ovr'),
    'Naive Bayes':         MultinomialNB(alpha=0.5),
}

emot_results = {}
print('🎭 Emotion Classification:')
for name, model in emotion_models.items():
    model.fit(X_train_tfidf_bi, y_emot_train)
    preds = model.predict(X_test_tfidf_bi)
    acc = accuracy_score(y_emot_test, preds)
    f1  = f1_score(y_emot_test, preds, average='weighted')
    emot_results[name] = {'Accuracy': acc, 'F1': f1, 'preds': preds}
    print(f'{name:<25} | Acc: {acc:.4f} | F1: {f1:.4f}')

best_emot_name = max(emot_results, key=lambda k: emot_results[k]['F1'])
best_emot_preds = emot_results[best_emot_name]['preds']

print(f'\n🏆 Best Emotion Model: {best_emot_name}')
print('\n📋 Detailed Report:')
print(classification_report(y_emot_test, best_emot_preds, target_names=le_emot.classes_))

In [ ]:
# Confusion Matrix Emotion
fig, ax = plt.subplots(figsize=(8, 7))
cm_emot = confusion_matrix(y_emot_test, best_emot_preds)
sns.heatmap(cm_emot, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_emot.classes_,
            yticklabels=le_emot.classes_,
            ax=ax, linewidths=0.5)
ax.set_title(f'Confusion Matrix - Emotion ({best_emot_name})', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrix_emotion.png', bbox_inches='tight', dpi=150)
plt.show()

## 📊 9. Final Summary & Insight

In [ ]:
print('='*60)
print('  PORTFOLIO SUMMARY - Indonesian Text Mining (J1)     ')
print('='*60)
print(f'\n📦 Dataset   : PRDECT-ID (Mendeley Data, CC BY 4.0)')
print(f'   Sumber    : Tokopedia product reviews (29 kategori)')
print(f'   Bahasa    : Bahasa Indonesia')
print(f'   Jumlah    : {len(df):,} reviews (setelah cleaning)')
print(f'\n🔧 Pipeline  :')
print(f'   ✅ Case folding + URL/mention removal')
print(f'   ✅ Slang normalization (custom dict)')
print(f'   ✅ Tokenisasi (NLTK)')
print(f'   ✅ Stopword removal (PySastrawi + custom)')
print(f'   ✅ Stemming Bahasa Indonesia (Sastrawi)')
print(f'\n🤖 Models    :')
for name, r in results.items():
    print(f'   {name:<35} F1={r["F1 (weighted)"]:.4f}')

best_f1 = results_df['F1 (weighted)'].max()
print(f'\n🏆 Best Sentiment F1: {best_f1:.4f} ({best_model_name})')
print(f'🏆 Best Emotion   F1: {emot_results[best_emot_name]["F1"]:.4f} ({best_emot_name})')

print(f'\n🔗 Relevansi ke Skripsi:')
print(f'   - Pipeline preprocessing ini IDENTIK dengan yang')
print(f'     dibutuhkan untuk berita finansial BEI')
print(f'   - Ganti domain: review produk → berita Bisnis Indonesia')
print(f'   - Ganti label: Love/Anger → Bullish/Bearish/Neutral')
print(f'   - Tambahkan: korelasi dengan data OHLCV dari yfinance')
print(f'   - IndoBERT fine-tuning = foundation untuk thesis model')
print('='*60)

## 🚀 10. Demo - Real-time Inference

In [ ]:
# Demo prediksi real-time dengan model terbaik
# (Bisa ganti teksnya sesuai kebutuhan)

best_model_for_inference = results[best_model_name]['model']

def predict_sentiment(text: str) -> dict:
    clean = preprocess_indonesian(text)
    vec   = tfidf_bi.transform([clean])
    pred  = best_model_for_inference.predict(vec)[0]
    label = le_sent.inverse_transform([pred])[0]
    return {'input': text, 'clean': clean, 'sentiment': label}

# Test samples - bisa diganti dengan berita finansial!
test_samples = [
    'Produknya sangat bagus dan pengiriman sangat cepat, saya sangat puas!',
    'Kecewa banget, barang tidak sesuai foto dan seller tidak responsif',
    'Lumayan lah, harganya sesuai dengan kualitas yang didapat',
    # Contoh kalimat domain finansial (preview ke skripsi)
    'IHSG menguat didorong sentimen positif dari laporan keuangan emiten',
    'Bursa saham Indonesia melemah akibat kekhawatiran inflasi global',
]

print('🔮 Real-time Sentiment Prediction:\n')
print(f'{"Input Text":<55} | Sentiment')
print('-'*70)
for sample in test_samples:
    result = predict_sentiment(sample)
    emoji = '🟢' if result['sentiment'] == 'Positive' else '🔴'
    print(f'{sample[:52]:<55} | {emoji} {result["sentiment"]}')

---
## 📝 Catatan & Referensi

### Dataset
- **PRDECT-ID**: Sutoyo, R., et al. (2022). *Product Reviews Dataset for Emotions Classification Tasks - Indonesian (PRDECT-ID) Dataset*. Mendeley Data. DOI: [10.17632/574v66hf2v.1](https://doi.org/10.17632/574v66hf2v.1). License: CC BY 4.0

### Model Referensi
- **IndoBERT**: Wilie, B., et al. (2020). *IndoNLU: Benchmark and Resources for Evaluating Indonesian Natural Language Understanding*. arXiv:2009.05387
- **PySastrawi**: Stemmer Bahasa Indonesia berbasis algoritma ECS (Enhanced Confix Stripping)

### Koneksi ke Skripsi
| Komponen Portfolio ini | Komponen Skripsi Finansial |
|---|---|
| Review produk Tokopedia | Berita finansial Bisnis Indonesia / Detik Finance |
| Label Positive/Negative | Label Bullish/Bearish/Neutral |
| IndoBERT fine-tuning | FinBERT Indonesia (novelty) |
| Accuracy, F1 | + Correlation dengan IHSG movement |
| EDA distribusi sentimen | + Time-series lag analysis |
